In [2]:
import pandas as pd

url = "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv"
df = pd.read_csv(url, storage_options={"User-Agent": "Our World In Data data fetch/1.0"})
df["date"] = pd.to_datetime(df["date"])
df.to_csv("../data/owid_raw.csv", index=False)

print(df.shape)
print(df["date"].min(), df["date"].max())
print(df.columns.tolist())
df.head()

(617667, 61)
2020-01-01 00:00:00 2026-08-30 00:00:00
['country', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'excess_mortality', 'excess_mortality_cumulative', 'excess_mortality_cumulative_absolute', 'excess_mortality_cumulative_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'icu_patients', 'icu_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'stringency_index', 'reproduction_rate', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed', 'new_tests_smoothed_per_thousand', 'positive_rate', 'tests_per_case', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_booster

,country,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,...,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand,human_development_index
0,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
1,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
2,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
3,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
4,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN


In [3]:
# 1. Check how aggregates (World, Europe, etc.) appear in this file
print("Rows with no continent:", df["continent"].isna().sum())
print(df.loc[df["continent"].isna(), "country"].unique())

# 2. Keep real countries only
countries = df[df["continent"].notna()].copy()

# 3. Keep only the columns we need
cols = ["country", "continent", "date",
        "new_cases_smoothed_per_million",
        "new_deaths_smoothed_per_million",
        "total_deaths_per_million",
        "people_fully_vaccinated_per_hundred",
        "stringency_index", "population", "median_age",
        "gdp_per_capita", "hospital_beds_per_thousand"]

focus = ["India", "United States", "Brazil", "United Kingdom",
         "Germany", "Japan", "South Africa", "Sweden"]

sub = countries.loc[countries["country"].isin(focus), cols].copy()
print("Countries found:", sorted(sub["country"].unique()))

# 4. Where does real death data actually start and end per country?
cov = (sub.dropna(subset=["new_deaths_smoothed_per_million"])
          .groupby("country")["date"].agg(["min", "max", "count"]))
print(cov)

# 5. Save the cleaned subset
sub.to_csv("../data/covid_focus.csv", index=False)
print(sub.shape)

Rows with no continent: 45240
<ArrowStringArray>
[                                             'Africa',
                                                'Asia',
                                    'Asia excl. China',
                                             'England',
                                   'England and Wales',
                                              'Europe',
                                 'European Union (27)',
                               'High-income countries',
                                'Low-income countries',
                       'Lower-middle-income countries',
                                       'North America',
                                    'Northern Ireland',
                                             'Oceania',
                                            'Scotland',
                                       'South America',
                                'Summer Olympics 2020',
                       'Upper-middle-income countries',